### <center> *Lab 03: Data Cleaning and Transformation Pipeline* </center>
---
**Student**: Emilio Daniel Guzmán Seda

In [1]:
from spark_utils import SparkUtils
spark_url = "spark://spark-master:7077"
app_name = "Spark SQL - Example 1"
su = SparkUtils(spark_url, app_name)
su

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/24 02:14:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local[*] appName=Spark SQL example>

In [2]:


columns_types = [("Order_ID", "long"),
                ("Company", "string"),
                ("City", "string"),
                ("Customer_Age", "int"),
                ("Order_Value", "float"),
                ("Delivery_Time_Min", "float"),
                ("Distance_Km", "float"),
                ("Items_Count", "float"),
                ("Product_Category", "string"),
                ("Payment_Method", "string"),
                ("Customer_Rating", "float"),
                ("Discount_Applied", "float"),
                ("Delivery_Partner_Rating", "float"),
                ]

qcommerce_schema = SparkUtils.generate_schema(columns_types)
#print(qcommerce_schema

qcommerce_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(qcommerce_schema) \
                .csv("/opt/spark/work-dir/data/quick_commerce_data_raw.csv/")

qcommerce_df.printSchema()
qcommerce_df.show(5)

root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+----

In [3]:
from pyspark.sql.functions import when, count, isnull, avg

qcommerce_df.columns

['Order_ID',
 'Company',
 'City',
 'Customer_Age',
 'Order_Value',
 'Delivery_Time_Min',
 'Distance_Km',
 'Items_Count',
 'Product_Category',
 'Payment_Method',
 'Customer_Rating',
 'Discount_Applied',
 'Delivery_Partner_Rating']

In [4]:
qcommerce_null_count_df = qcommerce_df.select([count(when(isnull(c), c)).alias(c) for c in qcommerce_df.columns])
qcommerce_null_count_df.show()

[Stage 1:===>                                                     (1 + 15) / 16]

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [5]:
qcommerce_null_count_df_2 = SparkUtils.count_nulls(qcommerce_df)
qcommerce_null_count_df_2.show()

[Stage 4:===>                                                     (1 + 15) / 16]

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [6]:
qcommerce_wo_nulls = qcommerce_df.dropna()
print(f"size of data frame after removing nulls: {qcommerce_wo_nulls.count()}")

[Stage 7:===>                                                     (1 + 15) / 16]

size of data frame after removing nulls: 780992


In [7]:
qcommerce_wo_nulls_fillna = qcommerce_df.fillna({
    'City': 'Uknown',
    'Items_Count': 0.0,
    'Customer_Rating': 0.0,
    'Delivery_Partner_Rating': 0.0
})
print(f"Size of data frame after removing nulls with 'fillna': {qcommerce_wo_nulls_fillna.count()}")

Size of data frame after removing nulls with 'fillna': 1000000


In [8]:
from pyspark.sql.functions import lit
qcommerce_wo_nulls_fillna = qcommerce_wo_nulls_fillna.withColumn("Code_Country", lit("IN"))
qcommerce_wo_nulls_fillna.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.

In [9]:
from pyspark.sql.functions import col
qcommerce_tax_df = qcommerce_wo_nulls_fillna.withColumn("Paid_Tax", col("Order_Value") * 0.16)
qcommerce_tax_df.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|        Paid_Tax|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN| 112.37400390625|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10

Task 1: Delivery SLA classification + filtering


* Create Delivery_SLA with withColumn + when:
  * Delivery_Time_Min ≤ 15 → "FAST", 15--20 → "ON_TIME", > 20 →
    "LATE"

* filter only Delivery_SLA = "LATE" and orderByDelivery_Time_Min(desc)

* Display: Order_ID, Company, City, Delivery_Time_Min, Delivery_SLA.

In [10]:
#Task One

qcommerce_df_task1 = qcommerce_tax_df.withColumn(
    "Delivery_SLA", 
    when(col("Delivery_Time_Min") <= 15, "FAST") \
    .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON-TIME") \
    .when(col("Delivery_Time_Min") > 20, "LATE")
).filter(col("Delivery_SLA") == "LATE") \
 .orderBy(col("Delivery_Time_Min"), ascending=False)

# Fixed the typo in the variable name here as well (tas1 -> task1)
qcommerce_df_task1.select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA").show()

[Stage 15:===>                                                    (1 + 15) / 16]

+--------+--------+--------+-----------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|
+--------+--------+--------+-----------------+------------+
| 1807126|Jio Mart|Haridwar|             40.0|        LATE|
| 1529779|Jio Mart|Haridwar|             40.0|        LATE|
| 1610720|Jio Mart|Haridwar|           39.994|        LATE|
| 1729503|Jio Mart|Haridwar|           39.994|        LATE|
| 1165764|Jio Mart|Haridwar|           39.994|        LATE|
| 1951122|Jio Mart|Haridwar|           39.988|        LATE|
| 1975896|Jio Mart|Haridwar|           39.988|        LATE|
| 1594835|Jio Mart|Haridwar|           39.982|        LATE|
| 1059671|Jio Mart|Haridwar|           39.982|        LATE|
| 1826295|Jio Mart|Haridwar|           39.982|        LATE|
| 1162804|Jio Mart|Haridwar|           39.982|        LATE|
| 1961544|Jio Mart|Haridwar|           39.976|        LATE|
| 1360875|Jio Mart|Haridwar|           39.964|        LATE|
| 1555058|Jio Mart|Haridwar|           3

Task 2: Discount impact (effective price + bucket + grouping)


* Create Effective_Order_Value = Order_Value * (1- Discount_Applied)

* Create Price_Bucket with when:

  * < 200→"LOW", 200--500→ "MEDIUM", > 500→ "HIGH"

* groupBy (Price_Bucket) and compute count(*) and avg(Effective_Order_Value).
* orderBy average effective value (desc).

In [11]:
#Task two

Effective_Order_Value = col("Order Value") * (1 - col("Discount_Applied"))

qcommerce_df_task2 = qcommerce_df_task1.withColumn(
    "Effective_Order_Value", col("Order_Value") * (1 - col("Discount_Applied"))
).withColumn(
    "Price_Bucket",
    when(col("Effective_Order_Value") < 200, "LOW") \
    .when((col("Effective_Order_Value") >= 200) & (col("Effective_Order_Value") <= 500), "MEDIUM") \
    .when(col("Effective_Order_Value") > 500, "HIGH")
).groupBy("Price_Bucket").agg(
     count("*").alias("Order_Count"),
     avg("Effective_Order_Value").alias("Avg_Effective_Value")
    ) \
    .orderBy(col("Avg_Effective_Value").desc())

qcommerce_df_task2.show(5)




[Stage 17:=======>                                                (2 + 14) / 16]

+------------+-----------+-------------------+
|Price_Bucket|Order_Count|Avg_Effective_Value|
+------------+-----------+-------------------+
|        HIGH|      56013|   707.219837039914|
|      MEDIUM|      59824| 353.49134017350553|
|         LOW|     143811|  25.36497016341368|
+------------+-----------+-------------------+



Task three: Customer segmentation (Age_Group) + category insights


* Create Age_Group with withColumn+when:
  * < 25→"YOUNG", 25--44→"ADULT", ≥ 45→"SENIOR"
* filter invalid ages (nulls, < 0, or > 100).

* groupBy(Age_Group, Product_Category)andcompute:
  * orders = count(*), avg_order_value = avg(Order_Value)
* orderBy(Age_Group asc, orders desc)tofindtopcategoriespersegment.

In [12]:
#Task three

qcommerce_df_task3 = qcommerce_tax_df.filter(
        col("Customer_Age").isNotNull() & 
        (col("Customer_Age") >= 0) & 
        (col("Customer_Age") <= 100)
    ).withColumn(
    "Age_Group",
    when(col("Customer_Age") < 25, "YOUNG") \
    .when((col("Customer_Age") >= 25) & (col("Customer_Age") <= 44), "ADULT") \
    .when(col("Customer_Age") >= 45, "SENIOR")
).groupBy("Age_Group", "Product_Category").agg(
     count("*").alias("orders"),
     avg("Order_Value").alias("Avg_Order_Value")
    ).orderBy(col("Age_Group").asc()) 

# 3. Show the result
qcommerce_df_task3.show(5)

[Stage 23:===>                                                    (1 + 15) / 16]

+---------+-------------------+------+-----------------+
|Age_Group|   Product_Category|orders|  Avg_Order_Value|
+---------+-------------------+------+-----------------+
|    ADULT|          Groceries| 68291|571.5250993844182|
|    ADULT|          Household| 68110|572.9259149188181|
|    ADULT|      Personal Care| 68331| 569.512671998805|
|    ADULT|              Dairy| 68512|573.4268899414931|
|    ADULT|Fruits & Vegetables| 67736|569.8593599596651|
+---------+-------------------+------+-----------------+
only showing top 5 rows


In [13]:
su._spark.stop()